# 04: Water Classification (Project 11)
**Name:** Devon Clermont-Gauer

Tests the lake / river / pond rules in `water_classify.py` on the 40 hand-corrected test tiles, so the rules can be checked and tuned before they are used in the dashboard.

**Rules:** river = elongation (centre-line length² / area) ≥ 10; otherwise pond if < 2 ha (Biggs et al., 2005), else lake. Only applied at 10–20 m resolution.

**How to run:** replace `YOUR-FOLDER` in Cell 1, then run all cells (no GPU needed).

In [ ]:
# Cell 1: connect Drive (test tiles) and load water_classify.py from the course repo
from google.colab import drive
drive.mount('/content/drive')

import os, sys, glob, subprocess
import numpy as np
import rasterio
import matplotlib.pyplot as plt

REPO = 'OSS4008A---Introduction-to-Remote-Sensing-and-Applications'
REPO_DIR = f'/content/{REPO}'
CODE_DIR = f'{REPO_DIR}/students/YOUR-FOLDER/code'      # replace YOUR-FOLDER
if os.path.isdir(f'{REPO_DIR}/.git'):
    subprocess.run(['git', '-C', REPO_DIR, 'pull', '--ff-only'], check=True)
else:
    subprocess.run(['git', 'clone', '--depth', '1', f'https://github.com/AdityaR28/{REPO}.git', REPO_DIR], check=True)
sys.path.insert(0, CODE_DIR)

import water_classify as wc
OUT  = '/content/drive/MyDrive/water_dataset'
CORR = f'{OUT}/test/masks_corrected'                     # hand-corrected masks (notebook 02)
tiles = sorted(glob.glob(f'{CORR}/*.tif'))
print('Corrected test tiles:', len(tiles))

In [ ]:
# Cell 2: classify every corrected test tile and summarise by region
def region_of(path):
    return os.path.basename(path).rsplit('_r', 1)[0]

def load_mask(path):
    with rasterio.open(path) as s:
        return s.read(1) == 1

totals = {}
for f in tiles:
    class_map, summary = wc.classify_water(load_mask(f), res=10)
    t = totals.setdefault(region_of(f), {n: 0 for n in ['lakes', 'rivers', 'ponds', 'partial']})
    for n in ['lakes', 'rivers', 'ponds']:
        t[n] += summary[n]['count']
    t['partial'] += summary['partial']

print(f"{'region':12} {'lakes':>6} {'rivers':>7} {'ponds':>6} {'partial':>8}")
for r, t in totals.items():
    print(f"{r:12} {t['lakes']:6} {t['rivers']:7} {t['ponds']:6} {t['partial']:8}")

In [ ]:
# Cell 3: look at the results: image | classes (blue = lake, cyan = river, green = pond)
import random
picks = random.Random(0).sample(tiles, min(6, len(tiles)))
fig, ax = plt.subplots(len(picks), 2, figsize=(8, 4 * len(picks)))
for row, f in enumerate(picks):
    name = os.path.basename(f)
    with rasterio.open(f'{OUT}/test/images/{name}') as s:
        rgb = np.clip(s.read([3, 2, 1]).transpose(1, 2, 0) / 3000, 0, 1)
    class_map, summary = wc.classify_water(load_mask(f), res=10)
    ax[row, 0].imshow(rgb); ax[row, 0].set_title(name, fontsize=9)
    ax[row, 1].imshow(rgb); ax[row, 1].imshow(wc.class_rgba(class_map))
    ax[row, 1].set_title(wc.summary_text(summary)[:60], fontsize=8)
    for a in ax[row]:
        a.axis('off')
plt.tight_layout()
plt.savefig('/content/classification_examples.png', dpi=100)
plt.show()

In [ ]:
# Cell 4: how sensitive is the river rule? Count rivers at different elongation thresholds
masks = [load_mask(f) for f in tiles]
print('threshold  rivers  lakes  ponds')
for thr in [6, 8, 10, 12, 15, 20]:
    wc.RIVER_ELONGATION = thr
    n = {'rivers': 0, 'lakes': 0, 'ponds': 0}
    for m in masks:
        s = wc.classify_water(m, 10)[1]
        for k in n:
            n[k] += s[k]['count']
    print(f"{thr:9}  {n['rivers']:6}  {n['lakes']:5}  {n['ponds']:5}")
wc.RIVER_ELONGATION = 10      # restore the default
# If many rivers in Cell 3 show as lakes, lower the threshold in water_classify.py;
# if branching lakes show as rivers, raise it.